In [ ]:
import win32gui
import win32con
import win32process
import subprocess
import time
import psutil
import mss
import numpy as np
import cv2
import pyautogui
import pygetwindow as gw
import dxcam

camera = dxcam.create(output_idx=1)
camera.start(target_fps=60, video_mode=True)

def capture_window_pyautogui(hwnd):
    """Capture the frame"""
    try:
        frame = camera.get_latest_frame()

        return cv2.resize(frame, (640, 480), interpolation=cv2.INTER_NEAREST)
        
        return frame
        
    except Exception as e:
        print(f"Error on capture: {e}")
        return None


def initialize_game():
    p = subprocess.Popen(
        r"F:\SteamLibrary\steamapps\common\Street Fighter 6\StreetFighter6.exe"
    )

    print("Starting the game...")

    hwnd = None
    timeout = 120
    start = time.time()

    while time.time() - start < timeout:
        hwnd = find_window_by_process_name("StreetFighter6")
        # if hwnd:
        #     print(f"Window found: hwnd={hwnd}")
        #     break
        if hwnd:
            print(f"Window found: hwnd={hwnd}")

            time.sleep(15)
            
            try:
                import ctypes
                user32 = ctypes.windll.user32
                
                monitor_info = (ctypes.c_ulong * 4)()
                ctypes.windll.user32.GetMonitorInfoW(1, monitor_info)
                monitor_left = monitor_info[0]
                monitor_top = monitor_info[1]
                monitor_right = monitor_info[2]
                monitor_bottom = monitor_info[3]
                
                win32gui.MoveWindow(hwnd, monitor_left, monitor_top, 
                                   monitor_right - monitor_left, 
                                   monitor_bottom - monitor_top, True)
                
                print(f"Moved window to monitor 1: {monitor_left},{monitor_top}")
                
            except Exception as e:
                print(f"Error moving window: {e}")
                
            break
        time.sleep(1)

    if not hwnd:
        raise RuntimeError("Could not find game window")

    # Hide the window
    # win32gui.ShowWindow(hwnd, win32con.SW_HIDE)
    print("Window hidden successfully.")
    
    return hwnd

def find_window_by_process_name(process_name):
    def callback(hwnd, result):
        if win32gui.IsWindowVisible(hwnd):
            tid, win_pid = win32process.GetWindowThreadProcessId(hwnd)
            try:
                process = psutil.Process(win_pid)
                if process_name.lower() in process.name().lower():
                    result.append(hwnd)
            except:
                pass
        return True

    result = []
    win32gui.EnumWindows(callback, result)
    return result[0] if result else None

hwnd = initialize_game()

try:
    while True:
        frame = capture_window_pyautogui(hwnd)

        cv2.imshow("", frame)
        cv2.waitKey(0)
        
        
        time.sleep(1/60)
        
except KeyboardInterrupt:
    print(f"Capture finished. Frames count: {env.frame_count}")

Starting the game...
Window found: hwnd=332142
Moved window to monitor 1: 0,0
Window hidden successfully.
